In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import os
import warnings

warnings.filterwarnings("ignore")

In [2]:
# ── Config ────────────────────────────────────────────────────────────────────
CSV_PATH   = "accounting_data.csv"   # change if your filename differs
OUTPUT_DIR = "eda_report"
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
# Palette — one color per account type, consistent across all charts
TYPE_COLORS = {
    "Asset":     "#378ADD",
    "Expense":   "#D85A30",
    "Revenue":   "#1D9E75",
    "Liability": "#BA7517",
}
BLUE   = "#378ADD"
GRAY   = "#888780"
RED    = "#E24B4A"
GREEN  = "#639922"
 
plt.rcParams.update({
    "font.family":     "sans-serif",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.color":         "#e8e8e8",
    "grid.linewidth":     0.6,
    "axes.labelsize":     11,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "figure.dpi":         130,
})

In [5]:
# ── Load & clean ───────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)

# Rename columns (snake_case, no spaces)
df.columns = [c.strip().replace(" ", "_").replace("(", "").replace(")", "")
              for c in df.columns]
 
df["Date"]     = pd.to_datetime(df["Date"])
df["Month"]    = df["Date"].dt.to_period("M").astype(str)
df["Quarter"]  = df["Date"].dt.to_period("Q").astype(str)
df["Success"]  = df["Transaction_Outcome"].map({1: "Success", 0: "Failure"})

# Derived columns
df["Net_Profit"] = df["Revenue"] - df["Expenditure"]
df["OpEx_Ratio"] = (df["Operating_Expenses"] / df["Revenue"]).round(4)
 
print(f"Loaded {len(df)} rows × {len(df.columns)} columns")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Missing_Data_Indicator flagged: {df['Missing_Data_Indicator'].sum()} rows")
print(f"Failed transactions: {(df['Transaction_Outcome'] == 0).sum()} rows\n")

# ── Helper ─────────────────────────────────────────────────────────────────────
def save(fig, name):
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  ✓ saved {name}")

Loaded 1000 rows × 23 columns
Date range: 2025-01-01 → 2027-09-27
Missing_Data_Indicator flagged: 48 rows
Failed transactions: 49 rows



In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# CHART 1 — Distribution overview (Transaction Amount, Cash Flow, Net Income,
#           Revenue, Expenditure, Profit Margin)
# ═══════════════════════════════════════════════════════════════════════════════
print("Chart 1: Distribution overview")
num_cols = ["Transaction_Amount", "Cash_Flow", "Net_Income",
            "Revenue", "Expenditure", "Profit_Margin"]
labels   = ["Transaction Amount", "Cash Flow", "Net Income",
            "Revenue", "Expenditure", "Profit Margin"]
 
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle("Distribution of Key Financial Metrics", fontsize=14, fontweight="bold", y=1.01)
 
for ax, col, label in zip(axes.flat, num_cols, labels):
    data = df[col]
    ax.hist(data, bins=30, color=BLUE, edgecolor="white", linewidth=0.5, alpha=0.85)
    ax.axvline(data.mean(),   color=RED,  linewidth=1.4, linestyle="--", label=f"Mean {data.mean():,.0f}")
    ax.axvline(data.median(), color=GRAY, linewidth=1.4, linestyle=":",  label=f"Median {data.median():,.0f}")
    ax.set_title(label, fontsize=10, fontweight="bold")
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")
    ax.legend(fontsize=7.5)
 
fig.tight_layout()
save(fig, "01_distributions.png")

Chart 1: Distribution overview
  ✓ saved 01_distributions.png


In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# CHART 2 — Transactions by Account Type (bar + donut side by side)
# ═══════════════════════════════════════════════════════════════════════════════
print("Chart 2: Account type breakdown")
type_counts = df["Account_Type"].value_counts()
colors      = [TYPE_COLORS[t] for t in type_counts.index]
 
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Transactions by Account Type", fontsize=14, fontweight="bold")
 
# Bar
bars = ax1.bar(type_counts.index, type_counts.values, color=colors, edgecolor="white", width=0.55)
for bar, val in zip(bars, type_counts.values):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
             f"{val}\n({val/len(df)*100:.1f}%)", ha="center", va="bottom", fontsize=9)
ax1.set_ylabel("Number of Transactions")
ax1.set_ylim(0, type_counts.max() * 1.18)
ax1.set_title("Count per type", fontsize=10)
 
# Donut
wedges, texts, autotexts = ax2.pie(
    type_counts.values, labels=type_counts.index, colors=colors,
    autopct="%1.1f%%", pctdistance=0.78, startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5}
)
for t in autotexts:
    t.set_fontsize(9)
centre_circle = plt.Circle((0, 0), 0.55, color="white")
ax2.add_artist(centre_circle)
ax2.set_title("Proportion", fontsize=10)
 
fig.tight_layout()
save(fig, "02_account_types.png")

Chart 2: Account type breakdown
  ✓ saved 02_account_types.png


In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# CHART 3 — Monthly Cash Flow trend line
# ═══════════════════════════════════════════════════════════════════════════════
print("Chart 3: Monthly cash flow trend")
monthly = (df.groupby("Month")
             .agg(Total_Cash_Flow=("Cash_Flow", "sum"),
                  Avg_Cash_Flow=("Cash_Flow", "mean"),
                  Transactions=("Transaction_ID", "count"))
             .reset_index())
 
fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(range(len(monthly)), monthly["Total_Cash_Flow"],
                alpha=0.12, color=BLUE)
ax.plot(range(len(monthly)), monthly["Total_Cash_Flow"],
        color=BLUE, linewidth=2, marker="o", markersize=4, label="Total cash flow")
 
# Rolling 7-period trend
roll = monthly["Total_Cash_Flow"].rolling(7, min_periods=1).mean()
ax.plot(range(len(monthly)), roll,
        color=RED, linewidth=1.5, linestyle="--", label="7-month rolling avg")
 
# Tick labels: every 4th month to avoid crowding
step = max(1, len(monthly) // 10)
ax.set_xticks(range(0, len(monthly), step))
ax.set_xticklabels(monthly["Month"].iloc[::step], rotation=30, ha="right")
ax.set_ylabel("Total Cash Flow")
ax.set_title("Monthly Total Cash Flow", fontsize=13, fontweight="bold")
ax.legend()
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))
 
fig.tight_layout()
save(fig, "03_monthly_cashflow.png")

Chart 3: Monthly cash flow trend
  ✓ saved 03_monthly_cashflow.png


In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# CHART 4 — Profit Margin by Account Type (boxplot + strip)
# ═══════════════════════════════════════════════════════════════════════════════
print("Chart 4: Profit margin by account type")
order = df.groupby("Account_Type")["Profit_Margin"].median().sort_values(ascending=False).index.tolist()
 
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x="Account_Type", y="Profit_Margin", order=order,
            palette=TYPE_COLORS, width=0.45, linewidth=1.2,
            flierprops={"marker": "o", "markersize": 3, "alpha": 0.4}, ax=ax)
sns.stripplot(data=df, x="Account_Type", y="Profit_Margin", order=order,
              palette=TYPE_COLORS, alpha=0.18, jitter=0.18, size=2.5, ax=ax)
 
# Annotate medians
for i, actype in enumerate(order):
    med = df[df["Account_Type"] == actype]["Profit_Margin"].median()
    ax.text(i, med + 0.012, f"{med:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
 
ax.set_title("Profit Margin by Account Type", fontsize=13, fontweight="bold")
ax.set_xlabel("Account Type")
ax.set_ylabel("Profit Margin")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
 
fig.tight_layout()
save(fig, "04_profit_margin_by_type.png")

Chart 4: Profit margin by account type
  ✓ saved 04_profit_margin_by_type.png


In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
# CHART 5 — Revenue vs Expenditure scatter (coloured by account type)
# ═══════════════════════════════════════════════════════════════════════════════
print("Chart 5: Revenue vs Expenditure scatter")
fig, ax = plt.subplots(figsize=(10, 7))
 
for actype, grp in df.groupby("Account_Type"):
    ax.scatter(grp["Revenue"], grp["Expenditure"],
               color=TYPE_COLORS[actype], alpha=0.55, s=22, label=actype, edgecolors="none")
 
# Break-even line
lim = max(df["Revenue"].max(), df["Expenditure"].max())
ax.plot([0, lim], [0, lim], color=GRAY, linewidth=1.2, linestyle="--",
        label="Break-even (Rev = Exp)", zorder=1)
 
ax.set_xlabel("Revenue")
ax.set_ylabel("Expenditure")
ax.set_title("Revenue vs Expenditure by Account Type", fontsize=13, fontweight="bold")
ax.legend(frameon=True, framealpha=0.85, fontsize=9)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))
 
fig.tight_layout()
save(fig, "05_revenue_vs_expenditure.png")

Chart 5: Revenue vs Expenditure scatter
  ✓ saved 05_revenue_vs_expenditure.png


In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# CHART 6 — Transaction Outcome (success vs failure) by Account Type
# ═══════════════════════════════════════════════════════════════════════════════
print("Chart 6: Success vs failure by account type")
outcome = (df.groupby(["Account_Type", "Success"])
             .size().unstack(fill_value=0)
             .reindex(columns=["Success", "Failure"]))
outcome_pct = outcome.div(outcome.sum(axis=1), axis=0) * 100
 
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Transaction Outcome by Account Type", fontsize=14, fontweight="bold")
 
# Absolute counts
outcome.plot(kind="bar", ax=ax1, color=[GREEN, RED], edgecolor="white",
             width=0.55, linewidth=0.8)
ax1.set_title("Absolute count", fontsize=10)
ax1.set_xlabel("")
ax1.set_ylabel("Transactions")
ax1.tick_params(axis="x", rotation=0)
ax1.legend(title="Outcome", fontsize=9)
for container in ax1.containers:
    ax1.bar_label(container, fontsize=8)
 
# Percentage
outcome_pct.plot(kind="bar", ax=ax2, color=[GREEN, RED], edgecolor="white",
                 width=0.55, linewidth=0.8)
ax2.set_title("Percentage", fontsize=10)
ax2.set_xlabel("")
ax2.set_ylabel("% of transactions")
ax2.tick_params(axis="x", rotation=0)
ax2.legend(title="Outcome", fontsize=9)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
for container in ax2.containers:
    ax2.bar_label(container, fmt="%.1f%%", fontsize=8)
 
fig.tight_layout()
save(fig, "06_outcome_by_type.png")

Chart 6: Success vs failure by account type
  ✓ saved 06_outcome_by_type.png


In [12]:
# ═══════════════════════════════════════════════════════════════════════════════
# CHART 7 — Correlation heatmap (numeric columns)
# ═══════════════════════════════════════════════════════════════════════════════
print("Chart 7: Correlation heatmap")
drop_cols = ["Transaction_ID", "Normalized_Transaction_Amount",
             "Missing_Data_Indicator", "Transaction_Outcome"]
num_df = df.select_dtypes("number").drop(columns=drop_cols, errors="ignore")
 
corr = num_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
 
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, linewidths=0.4, linecolor="#f0f0f0",
            annot_kws={"size": 8}, ax=ax,
            cbar_kws={"shrink": 0.7, "label": "Pearson r"})
ax.set_title("Correlation Matrix — Numeric Features", fontsize=13, fontweight="bold")
ax.tick_params(axis="x", rotation=30)
 
fig.tight_layout()
save(fig, "07_correlation_heatmap.png")

Chart 7: Correlation heatmap
  ✓ saved 07_correlation_heatmap.png


In [13]:
# ═══════════════════════════════════════════════════════════════════════════════
# CHART 8 — Processing Time distribution + accuracy by outcome
# ═══════════════════════════════════════════════════════════════════════════════
print("Chart 8: Processing time & accuracy score")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Operational Metrics", fontsize=14, fontweight="bold")
 
# Processing time histogram split by outcome
for outcome_val, label, color in [(1, "Success", GREEN), (0, "Failure", RED)]:
    subset = df[df["Transaction_Outcome"] == outcome_val]["Processing_Time_seconds"]
    ax1.hist(subset, bins=25, color=color, alpha=0.6, label=f"{label} (n={len(subset)})",
             edgecolor="white", linewidth=0.5)
ax1.axvline(df["Processing_Time_seconds"].mean(), color=GRAY, linewidth=1.5,
            linestyle="--", label=f"Overall mean {df['Processing_Time_seconds'].mean():.2f}s")
ax1.set_title("Processing Time distribution", fontsize=10)
ax1.set_xlabel("Processing Time (seconds)")
ax1.set_ylabel("Count")
ax1.legend(fontsize=8.5)
 
# Accuracy score by account type (violin)
sns.violinplot(data=df, x="Account_Type", y="Accuracy_Score",
               palette=TYPE_COLORS, inner="quartile", linewidth=1, ax=ax2)
ax2.set_title("Accuracy Score by Account Type", fontsize=10)
ax2.set_xlabel("Account Type")
ax2.set_ylabel("Accuracy Score")
 
fig.tight_layout()
save(fig, "08_operational_metrics.png")

Chart 8: Processing time & accuracy score
  ✓ saved 08_operational_metrics.png


In [14]:
# ═══════════════════════════════════════════════════════════════════════════════
# CHART 9 — Debt-to-Equity Ratio over time (line + threshold)
# ═══════════════════════════════════════════════════════════════════════════════
print("Chart 9: Debt-to-equity ratio trend")
de_monthly = (df.groupby("Month")["Debt-to-Equity_Ratio"]
                .mean()
                .reset_index()
                .rename(columns={"Debt-to-Equity_Ratio": "avg_DE"}))
 
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(len(de_monthly)), de_monthly["avg_DE"],
        color=BLUE, linewidth=2, marker="o", markersize=4)
ax.fill_between(range(len(de_monthly)), de_monthly["avg_DE"],
                alpha=0.1, color=BLUE)
 
# Risk threshold line at D/E = 2
ax.axhline(2.0, color=RED, linewidth=1.4, linestyle="--", label="High-risk threshold (D/E = 2.0)")
 
step = max(1, len(de_monthly) // 10)
ax.set_xticks(range(0, len(de_monthly), step))
ax.set_xticklabels(de_monthly["Month"].iloc[::step], rotation=30, ha="right")
ax.set_ylabel("Avg Debt-to-Equity Ratio")
ax.set_title("Monthly Average Debt-to-Equity Ratio", fontsize=13, fontweight="bold")
ax.legend()
 
fig.tight_layout()
save(fig, "09_de_ratio_trend.png")

Chart 9: Debt-to-equity ratio trend
  ✓ saved 09_de_ratio_trend.png


In [15]:
# ═══════════════════════════════════════════════════════════════════════════════
# CHART 10 — Missing data & flagged rows summary
# ═══════════════════════════════════════════════════════════════════════════════
print("Chart 10: Data quality summary")
missing_by_type = (df[df["Missing_Data_Indicator"] == 1]
                     .groupby("Account_Type").size()
                     .reindex(df["Account_Type"].unique(), fill_value=0)
                     .sort_values(ascending=True))
 
total_by_type = df.groupby("Account_Type").size()
pct_missing   = (missing_by_type / total_by_type * 100).round(1)
 
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Data Quality — Missing Data Flag", fontsize=14, fontweight="bold")
 
# Count
colors_bar = [TYPE_COLORS[t] for t in missing_by_type.index]
bars = ax1.barh(missing_by_type.index, missing_by_type.values,
                color=colors_bar, edgecolor="white", height=0.5)
for bar, val in zip(bars, missing_by_type.values):
    ax1.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
             str(val), va="center", fontsize=9)
ax1.set_title("Flagged rows per account type", fontsize=10)
ax1.set_xlabel("Count")
ax1.set_xlim(0, missing_by_type.max() * 1.2)
 
# % rate
bars2 = ax2.barh(pct_missing.index, pct_missing.values,
                 color=colors_bar, edgecolor="white", height=0.5)
for bar, val in zip(bars2, pct_missing.values):
    ax2.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
             f"{val}%", va="center", fontsize=9)
ax2.set_title("% flagged within each type", fontsize=10)
ax2.set_xlabel("% of type")
ax2.set_xlim(0, pct_missing.max() * 1.25)
ax2.xaxis.set_major_formatter(mtick.PercentFormatter())
 
fig.tight_layout()
save(fig, "10_data_quality.png")

Chart 10: Data quality summary
  ✓ saved 10_data_quality.png


In [16]:
# ═══════════════════════════════════════════════════════════════════════════════
# TEXT SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 55)
print("EDA SUMMARY")
print("═" * 55)
print(f"Total rows          : {len(df):,}")
print(f"Date range          : {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Unique account types: {df['Account_Type'].nunique()}")
print(f"Avg profit margin   : {df['Profit_Margin'].mean():.2%}")
print(f"Success rate        : {df['Transaction_Outcome'].mean():.2%}")
print(f"Missing data flags  : {df['Missing_Data_Indicator'].sum()} ({df['Missing_Data_Indicator'].mean():.2%})")
print(f"Avg D/E ratio       : {df['Debt-to-Equity_Ratio'].mean():.2f}")
print(f"Avg processing time : {df['Processing_Time_seconds'].mean():.2f}s")
print(f"\nAll charts saved to: ./{OUTPUT_DIR}/")
print("═" * 55)


═══════════════════════════════════════════════════════
EDA SUMMARY
═══════════════════════════════════════════════════════
Total rows          : 1,000
Date range          : 2025-01-01 → 2027-09-27
Unique account types: 4
Avg profit margin   : 49.91%
Success rate        : 95.10%
Missing data flags  : 48 (4.80%)
Avg D/E ratio       : 1.78
Avg processing time : 1.24s

All charts saved to: ./eda_report/
═══════════════════════════════════════════════════════
